<a href="https://colab.research.google.com/github/Steph-business/Tech_Talent_Accelerator/blob/main/week6_day3_dalyckallenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 0. Préparation de l'environnement
Nous commençons par installer les bibliothèques nécessaires et télécharger le jeu de données fourni.

In [ ]:
!pip install -q transformers datasets
!wget https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/Week%206/W6D1%20GenAi%20France/Basics%20of%20BERT%20and%20XLM-RoBERTa%20-%20PyTorch%20-%202.zip -O dataset.zip
!unzip -o dataset.zip

### 1 & 2. Compréhension et Tokenisation
Nous allons charger les tokeniseurs pour BERT et XLM-RoBERTa afin d'observer la différence de traitement des textes.

In [ ]:
from transformers import BertTokenizer, XLMRobertaTokenizer
import torch

# Chargement des tokeniseurs
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
xlm_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')

text = "L'apprentissage profond est fascinant."

# Exemple de tokenisation
bert_input = bert_tokenizer.encode_plus(text, add_special_tokens=True, max_length=15, padding='max_length', truncation=True, return_tensors='pt')
xlm_input = xlm_tokenizer.encode_plus(text, add_special_tokens=True, max_length=15, padding='max_length', truncation=True, return_tensors='pt')

print("BERT Tokens:", bert_tokenizer.tokenize(text))
print("BERT IDs:", bert_input['input_ids'])
print("\nXLM-RoBERTa Tokens:", xlm_tokenizer.tokenize(text))
print("XLM IDs:", xlm_input['input_ids'])

# Décodage pour voir les jetons spéciaux
print("\nDécodage BERT:", bert_tokenizer.decode(bert_input['input_ids'][0]))
print("Décodage XLM:", xlm_tokenizer.decode(xlm_input['input_ids'][0]))

### 4. Chargement et exploration de l'ensemble de données
Nous allons maintenant charger les fichiers CSV extraits pour analyser leur structure.

In [3]:
import pandas as pd
import os
import glob

# Recherche récursive de tous les fichiers CSV
csv_files = glob.glob('**/*.csv', recursive=True)
print("Fichiers CSV trouvés :", csv_files)

if not csv_files:
    print("Aucun CSV trouvé. Création d'un dataset factice pour la démonstration.")
    data = {
        'text': [f"Exemple de phrase {i}" for i in range(100)],
        'label': [0, 1] * 50
    }
    df = pd.DataFrame(data)
else:
    try:
        df = pd.read_csv(csv_files[0])
        print(f"Chargement de : {csv_files[0]}")
    except Exception as e:
        print(f"Erreur de lecture : {e}")

if 'df' in locals():
    display(df.head())
    print("Forme du dataset :", df.shape)

Fichiers CSV trouvés : ['sample_data/mnist_train_small.csv', 'sample_data/mnist_test.csv', 'sample_data/california_housing_test.csv', 'sample_data/california_housing_train.csv']
Chargement de : sample_data/mnist_train_small.csv


,6,0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,...,0.581,0.582,0.583,0.584,0.585,0.586,0.587,0.588,0.589,0.590
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Forme du dataset : (19999, 785)


### 5. Création de plis de validation croisée
Nous utilisons `StratifiedKFold` pour diviser nos données en 5 plis, en veillant à ce que chaque pli conserve la même proportion de classes que l'ensemble de données complet.

In [4]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

if 'df' in locals():
    try:
        # Configuration de la validation croisée
        kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

        # Identification de la colonne cible (souvent 'label' ou la dernière colonne)
        target_column = 'label' if 'label' in df.columns else df.columns[-1]
        print(f"Colonne cible utilisée pour la stratification : {target_column}")

        train_folds = []
        val_folds = []

        # Séparation en plis
        for fold, (train_idx, val_idx) in enumerate(kf.split(df, df[target_column])):
            train_folds.append(df.iloc[train_idx])
            val_folds.append(df.iloc[val_idx])
            print(f"Plis {fold+1}: Train={len(train_idx)}, Val={len(val_idx)}")

        print("\nValidation croisée configurée avec succès.")
    except Exception as e:
        print(f"Erreur lors de la validation croisée : {e}")
else:
    print("Erreur : Le DataFrame 'df' n'est pas disponible.")

Colonne cible utilisée pour la stratification : 0.590
Plis 1: Train=15999, Val=4000
Plis 2: Train=15999, Val=4000
Plis 3: Train=15999, Val=4000
Plis 4: Train=15999, Val=4000
Plis 5: Train=16000, Val=3999

Validation croisée configurée avec succès.
